# MPC 기반 유도법칙 연구 (Model Predictive Control Guidance)

## 비례항법을 넘어서 — 최적 유도를 실시간으로 구현할 수 있을까?

---

> 이 노트북은 개인 연구로 MPC 유도를 탐구한 내용입니다.  
> 석사 과정에서 MPC 기반 유도에 관심을 갖고 공부한 내용을 정리했으며,  
> 시행착오를 포함해 탐색적으로 기록했습니다.

비례항법(Proportional Navigation, PN)은 수십 년간 유도탄 유도의 표준으로 자리잡아 왔다.  
단순하고 강인하며, 계산 부담이 거의 없다는 장점 때문이다.  
하지만 PN에는 근본적인 한계가 있다 — 제약조건을 고려하지 않는다는 것이다.

**가속도가 포화(saturation)되면 어떻게 할 것인가?**  
**Seeker의 시야각(FOV) 제한이 있다면?**  
**표적이 기동한다면 미리 예측할 수 있을까?**

이런 질문에서 출발해 MPC(Model Predictive Control) 유도를 공부하기 시작했다.  
참고: Zarchan, *Tactical and Strategic Missile Guidance* (Ch. 8), Boyd et al., *Convex Optimization*

---
## 1. 왜 MPC인가?

### 1.1 비례항법(PN)의 한계

PN 유도법칙은 다음과 같이 정의된다:

$$a_{cmd} = N \cdot V_c \cdot \dot{\lambda}$$

- $N$: 유효 항법 상수 (보통 3~5)
- $V_c$: 폐색 속도 (closing velocity)
- $\dot{\lambda}$: LOS(Line-of-Sight) 각속도

PN의 문제점:

| 한계 | 설명 |
|------|------|
| 제약조건 미고려 | 가속도 한계 $|a| \leq a_{max}$를 명시적으로 처리하지 않음 |
| 반응적(reactive) | 현재 LOS rate에만 반응, 미래 예측 없음 |
| FOV 무시 | Seeker 시야각 제한을 고려하지 않음 |
| 표적 기동 | 기동 표적에 대한 적응 능력 제한적 (APN으로 일부 보완) |

### 1.2 MPC의 장점

MPC는 미래 $N$ 스텝을 예측하면서 최적 제어 입력 시퀀스를 계산한다:

$$\min_{u_0, u_1, \ldots, u_{N-1}} \sum_{k=0}^{N-1} \ell(x_k, u_k) + V_f(x_N)$$

$$\text{subject to: } x_{k+1} = f(x_k, u_k), \quad |u_k| \leq a_{max}$$

- **예측 기반 최적화**: 미래 $N$ 스텝을 내다보고 최적 가속도 시퀀스 계산
- **제약조건 명시적 처리**: 가속도 한계, FOV 등을 최적화 문제에 직접 포함
- **Receding horizon**: 첫 번째 제어 입력만 적용하고 다음 스텝에서 재최적화

### 1.3 실시간 계산 문제

MPC의 가장 큰 걸림돌은 계산 시간이다.  
유도탄 탑재 컴퓨터에서 $dt = 10\,\text{ms}$ 내에 비선형 최적화를 풀어야 한다.

> **처음 접근**: CasADi + IPOPT로 NLP(Nonlinear Program) 풀기  
> **이 노트북**: 먼저 scipy로 개념을 검증해보자  
> **결론**: 실시간 적용을 위해서는 추가 연구 필요

---
## 2. LOS-Relative 상태 공간

### 2.1 왜 LOS 좌표계인가?

NED(North-East-Down) 좌표계로 교전을 기술하면 상태 공간이 불필요하게 복잡해진다.  
LOS 상대 좌표계에서는 유도 성능 지표(miss distance)와 직결되는 변수들로 상태를 구성할 수 있다.

**상태 벡터 (6차원)**:
$$\mathbf{x} = [R, \; V_c, \; \dot{\lambda}_{az}, \; \dot{\lambda}_{el}, \; a_{pitch,ach}, \; a_{yaw,ach}]^T$$

| 변수 | 의미 | 단위 |
|------|------|------|
| $R$ | 유도탄-표적 거리 | m |
| $V_c$ | 폐색 속도 ($V_c = -\dot{R}$, 접근 시 양수) | m/s |
| $\dot{\lambda}_{az}$ | LOS 방위각 속도 | rad/s |
| $\dot{\lambda}_{el}$ | LOS 고각 속도 | rad/s |
| $a_{pitch,ach}$ | 달성된 피치 가속도 | m/s² |
| $a_{yaw,ach}$ | 달성된 요 가속도 | m/s² |

**제어 입력 (2차원)**:
$$\mathbf{u} = [a_{pitch,cmd}, \; a_{yaw,cmd}]^T$$

### 2.2 LOS 각속도 동역학 — Zarchan Ch. 8

Zarchan의 횡방향(transverse) 방정식 유도:

상대 운동 벡터 $\mathbf{R} = \mathbf{r}_T - \mathbf{r}_M$를 미분하면:

$$\ddot{\mathbf{R}} = \mathbf{a}_T - \mathbf{a}_M$$

LOS 방향 단위벡터 $\hat{R}$와 횡방향 성분으로 분리하면 (Ch. 8, eq. 8.x):

$$R \ddot{\lambda} + 2 \dot{R} \dot{\lambda} = a_T^{\perp} - a_M^{\perp}$$

$V_c = -\dot{R}$ (접근 시 양수)로 정의하면 $\dot{R} = -V_c$이므로:

$$R \ddot{\lambda} - 2 V_c \dot{\lambda} = a_T^{\perp} - a_M^{\perp}$$

$$\boxed{\ddot{\lambda} = \frac{2 V_c \dot{\lambda} + a_T^{\perp} - a_M^{\perp}}{R}}$$

> **부호 규약 주의**: $+2V_c\dot{\lambda}$ 항이 **양수**다!  
> 이 부분에서 한참 헤맸다. 처음에 부호를 반대로 쓰다가 시뮬레이션이 발산하는 현상을 겪었다.  
> Zarchan 원문을 꼼꼼히 읽고, $V_c = -\dot{R}$의 부호 정의를 정확히 확인해야 했다.  
> **Coriolis 항 $+2V_c\dot{\lambda}$은 물리적으로 타당하다**: 거리가 줄어들면($V_c > 0$) LOS rate가 증폭되는 kinematic amplification 효과.

### 2.3 완전한 상태 방정식

$$\dot{R} = -V_c$$

$$\dot{V}_c = -\dot{\lambda}^2 R - a_{T,radial}$$

$$\ddot{\lambda}_{az} = \frac{2 V_c \dot{\lambda}_{az} + a_{T,az} - a_{yaw,ach}}{R}$$

$$\ddot{\lambda}_{el} = \frac{2 V_c \dot{\lambda}_{el} + a_{T,el} - a_{pitch,ach}}{R}$$

$$\dot{a}_{pitch,ach} = \frac{a_{pitch,cmd} - a_{pitch,ach}}{\tau_{ap}}$$

$$\dot{a}_{yaw,ach} = \frac{a_{yaw,cmd} - a_{yaw,ach}}{\tau_{ap}}$$

마지막 두 식은 자동조종장치(autopilot)의 1차 지연을 모델링한다. $\tau_{ap}$는 보통 50ms.

In [ ]:
"""Cell 1: LOS-relative dynamics (self-contained, numpy only)"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────
# LOS-Relative Dynamics (numpy, no CasADi)
# Reference: Zarchan Ch.8, LOSRelativeDynamics in los_relative_dynamics.py
# ─────────────────────────────────────────────────────────────────

R_MIN = 0.1  # singularity guard (m)

def los_derivatives(x, u, tau_ap=0.05, n_T_az=0.0, n_T_el=0.0, a_T_radial=0.0):
    """LOS-relative state derivative.

    State x = [R, Vc, lam_dot_az, lam_dot_el, a_pitch_ach, a_yaw_ach]
    Control u = [a_pitch_cmd, a_yaw_cmd]

    부호 규약 (Zarchan Ch.8):
        Vc = -dR/dt  (양수 = 접근 중)
        lam_ddot = (2*Vc*lam_dot + aT - aM) / R   <-- Coriolis 항 양수!
    """
    R        = x[0]
    Vc       = x[1]
    ldot_az  = x[2]
    ldot_el  = x[3]
    ap_ach   = x[4]
    ay_ach   = x[5]

    ap_cmd   = u[0]
    ay_cmd   = u[1]

    R_safe   = max(R, R_MIN)

    # 거리 변화율
    dR = -Vc

    # 폐색 속도 변화율: 원심 감속 + 표적 축방향 가속
    lam_dot_sq = ldot_az**2 + ldot_el**2
    dVc = -lam_dot_sq * R_safe - a_T_radial

    # LOS 각속도 동역학 (Zarchan Ch.8 횡방향 방정식)
    # R*lambda_ddot + 2*R_dot*lambda_dot = aT - aM
    # => lambda_ddot = (2*Vc*lambda_dot + aT - aM) / R   [Vc = -R_dot]
    d_ldot_az = (2.0 * Vc * ldot_az + n_T_az - ay_ach) / R_safe
    d_ldot_el = (2.0 * Vc * ldot_el + n_T_el - ap_ach) / R_safe

    # 자동조종장치 1차 지연
    d_ap_ach = (ap_cmd - ap_ach) / tau_ap
    d_ay_ach = (ay_cmd - ay_ach) / tau_ap

    return np.array([dR, dVc, d_ldot_az, d_ldot_el, d_ap_ach, d_ay_ach])


def los_rk4_step(x, u, dt, tau_ap=0.05, n_T_az=0.0, n_T_el=0.0, a_T_radial=0.0):
    """4차 Runge-Kutta 한 스텝 적분."""
    kwargs = dict(tau_ap=tau_ap, n_T_az=n_T_az, n_T_el=n_T_el, a_T_radial=a_T_radial)
    k1 = los_derivatives(x,                u, **kwargs)
    k2 = los_derivatives(x + 0.5*dt*k1,   u, **kwargs)
    k3 = los_derivatives(x + 0.5*dt*k2,   u, **kwargs)
    k4 = los_derivatives(x +      dt*k3,  u, **kwargs)
    return x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def ned_to_los_state(r_M, v_M, r_T, v_T, a_pitch_ach=0.0, a_yaw_ach=0.0):
    """NED 상태를 LOS-relative 상태 벡터로 변환."""
    r_M, v_M = np.asarray(r_M, float), np.asarray(v_M, float)
    r_T, v_T = np.asarray(r_T, float), np.asarray(v_T, float)

    R_vec  = r_T - r_M
    Rd_vec = v_T - v_M
    R      = float(np.linalg.norm(R_vec))

    if R < 1e-6:
        return np.array([R, 0.0, 0.0, 0.0, a_pitch_ach, a_yaw_ach])

    R_hat  = R_vec / R
    Vc     = float(-np.dot(R_hat, Rd_vec))  # 양수 = 접근 중

    Rx, Ry, Rz   = R_vec
    Rdx, Rdy, _  = Rd_vec
    rho_h        = float(np.sqrt(Rx**2 + Ry**2))

    if rho_h < 1e-6:
        Omega    = np.cross(R_vec, Rd_vec) / (R * R)
        lam_dot_az = 0.0
        lam_dot_el = float(np.sqrt(Omega[0]**2 + Omega[1]**2))
    else:
        lam_dot_az = float((Rx*Rdy - Ry*Rdx) / rho_h**2)
        lam_dot_el = float(
            (-Rd_vec[2]*rho_h**2 + Rz*(Rx*Rdx + Ry*Rdy)) / (R**2 * rho_h)
        )

    return np.array([R, Vc, lam_dot_az, lam_dot_el, float(a_pitch_ach), float(a_yaw_ach)])


# 빠른 확인: 초기 상태 변환 테스트
r_M_test = np.array([0.0,    0.0,    -5000.0])  # NED (m)
v_M_test = np.array([300.0,  0.0,    0.0])
r_T_test = np.array([10000.0, 500.0, -5000.0])
v_T_test = np.array([-200.0,  0.0,    0.0])

x0_test = ned_to_los_state(r_M_test, v_M_test, r_T_test, v_T_test)
labels  = ['R(m)', 'Vc(m/s)', 'lam_dot_az(rad/s)', 'lam_dot_el(rad/s)',
           'a_pitch_ach(m/s²)', 'a_yaw_ach(m/s²)']

print("LOS-Relative 초기 상태:")
for lb, val in zip(labels, x0_test):
    print(f"  {lb:30s} = {val:12.4f}")

---
## 3. 최적화 문제 정의

### 3.1 비용 함수 (Cost Function)

MPC는 다음 비용 함수를 최소화한다:

$$J = \underbrace{w_{terminal} \cdot (\|ZEM_N\|^2)}_{\text{종단 miss distance}} + \sum_{k=0}^{N-1} \left[ \underbrace{w_{effort} \cdot \|u_k\|^2}_{\text{제어 에너지}} + \underbrace{w_{rate} \cdot \|\Delta u_k\|^2}_{\text{제어 변화율}} \right]$$

**ZEM (Zero-Effort Miss)**: 제어 입력 없이 현재 상태를 유지했을 때의 예측 miss distance

$$ZEM_{az} \approx R \cdot \dot{\lambda}_{az} \cdot t_{go} + \frac{1}{2} a_{T,az} \cdot t_{go}^2$$

$$ZEM_{el} \approx R \cdot \dot{\lambda}_{el} \cdot t_{go} + \frac{1}{2} a_{T,el} \cdot t_{go}^2$$

여기서 $t_{go} = R / V_c$ (남은 비행 시간 추정).

### 3.2 제약조건

$$|a_{pitch,cmd}| \leq a_{max}, \quad |a_{yaw,cmd}| \leq a_{max}$$

일반적으로 $a_{max} = 40g = 392.4\,\text{m/s}^2$ (최대 가속도 한계).

### 3.3 가중치 튜닝 경험

> **처음 시도**: $w_{effort} = 1.0$ (제어 에너지에 큰 페널티)  
> **결과**: miss distance가 크게 나왔다. 유도탄이 "연료 절약"을 너무 중시해서 표적을 놓침  
>
> **수정**: $w_{effort} = 0.0$, $w_{terminal} = 100.0$ (종단 miss를 크게 페널티)  
> **결과**: miss distance가 현저히 감소  
>
> **최종 설정**: $w_{effort}=0.0$, $w_{rate}=0.005$, $w_{terminal}=100.0$  
> - 제어 에너지 페널티 제거: 최대 40g 권위(authority) 모두 활용 가능
> - 작은 rate 페널티: 명령 oscillation 방지
> - 큰 terminal 가중치: miss → 0 구동력

In [ ]:
"""Cell 2: Shooting method MPC (scipy.optimize.minimize 사용)

CasADi 없이 먼저 scipy로 시도해봄.
Simple shooting: 전체 제어 시퀀스 [u_0, ..., u_{N-1}]를 한 번에 최적화.
"""

# ─────────────────────────────────────────────────────────────────
# ZEM 계산
# ─────────────────────────────────────────────────────────────────

def compute_zem(x, n_T_az=0.0, n_T_el=0.0):
    """Zero-Effort Miss 계산 (linear propagation)."""
    R, Vc, ldot_az, ldot_el = x[0], x[1], x[2], x[3]
    Vc_safe = max(Vc, 1.0)
    t_go    = R / Vc_safe
    zem_az  = R * ldot_az * t_go + 0.5 * n_T_az * t_go**2
    zem_el  = R * ldot_el * t_go + 0.5 * n_T_el * t_go**2
    return zem_az, zem_el, t_go


# ─────────────────────────────────────────────────────────────────
# MPC 비용 함수
# ─────────────────────────────────────────────────────────────────

def mpc_cost(u_flat, x0, N, dt_mpc, tau_ap, a_max,
             w_terminal=100.0, w_effort=0.0, w_rate=0.005,
             n_T_az=0.0, n_T_el=0.0):
    """MPC 비용 함수 (shooting method).

    u_flat: 길이 2*N의 1D 배열 [u0_pitch, u0_yaw, u1_pitch, u1_yaw, ...]
    """
    U   = u_flat.reshape(N, 2)   # (N, 2)
    J   = 0.0
    x   = x0.copy()
    u_prev = np.zeros(2)

    for k in range(N):
        u_k  = U[k]
        # 제어 변화율 페널티
        du   = u_k - u_prev
        J   += w_rate   * float(np.dot(du, du))
        J   += w_effort * float(np.dot(u_k, u_k))
        u_prev = u_k

        # 상태 전파
        x = los_rk4_step(x, u_k, dt_mpc, tau_ap=tau_ap,
                         n_T_az=n_T_az, n_T_el=n_T_el)

    # 종단 비용: ZEM^2
    zem_az, zem_el, _ = compute_zem(x, n_T_az, n_T_el)
    J += w_terminal * (zem_az**2 + zem_el**2)

    return J


# ─────────────────────────────────────────────────────────────────
# MPC 한 스텝 풀기
# ─────────────────────────────────────────────────────────────────

def mpc_step(x0, N=10, dt_mpc=0.1, tau_ap=0.05, a_max=392.4,
             w_terminal=100.0, w_effort=0.0, w_rate=0.005,
             n_T_az=0.0, n_T_el=0.0, u_warm=None):
    """scipy.optimize.minimize로 MPC 한 스텝 최적화.

    Returns:
        (a_pitch_cmd, a_yaw_cmd): 첫 번째 스텝 제어 입력
        u_opt (N,2): 전체 최적 제어 시퀀스 (다음 warm start용)
    """
    # 초기 추정값
    if u_warm is not None and u_warm.shape == (N, 2):
        u0_flat = u_warm.flatten()
    else:
        u0_flat = np.zeros(2 * N)

    # 제약조건 (bound로 처리)
    bounds = [(-a_max, a_max)] * (2 * N)

    result = minimize(
        mpc_cost,
        u0_flat,
        args=(x0, N, dt_mpc, tau_ap, a_max,
              w_terminal, w_effort, w_rate, n_T_az, n_T_el),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 100, 'ftol': 1e-9}
    )

    u_opt = result.x.reshape(N, 2)
    return float(u_opt[0, 0]), float(u_opt[0, 1]), u_opt


print("MPC 최적화 함수 정의 완료")
print(f"  최적화 방법: scipy L-BFGS-B (bound constraints)")
print(f"  결정 변수: 2 * N = 2 * 10 = 20 (pitch/yaw x horizon)")
print(f"  가중치: w_effort=0.0, w_rate=0.005, w_terminal=100.0")

In [ ]:
"""Cell 3: MPC 교전 시뮬레이션 (2D, 단순화)"""

matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False


def run_2d_engagement_mpc(dt=0.01, t_max=30.0,
                          missile_pos=None, missile_vel=None,
                          target_pos=None, target_vel=None,
                          a_max=392.4, tau_ap=0.05,
                          guidance='mpc',
                          mpc_N=10, mpc_dt=0.1,
                          mpc_w_terminal=100.0, mpc_w_effort=0.0, mpc_w_rate=0.005,
                          pn_N=4.0):
    """2D 단순화 교전 시뮬레이션.

    guidance: 'mpc' 또는 'apn'
    NED 평면 (x=North, z=Down), y=0으로 단순화.
    """
    if missile_pos is None:
        missile_pos = np.array([0.0, -5000.0])    # [x_north, z_down]
    if missile_vel is None:
        missile_vel = np.array([300.0, 0.0])
    if target_pos is None:
        target_pos  = np.array([10000.0, -5000.0])
    if target_vel is None:
        target_vel  = np.array([-200.0, 0.0])

    r_M = np.array([missile_pos[0], 0.0, missile_pos[1]])  # NED 3D
    v_M = np.array([missile_vel[0], 0.0, missile_vel[1]])
    r_T = np.array([target_pos[0],  0.0, target_pos[1]])
    v_T = np.array([target_vel[0],  0.0, target_vel[1]])

    # 기록용
    t_hist, rM_hist, rT_hist = [], [], []
    acmd_hist, range_hist    = [], []
    lam_dot_hist             = []

    a_pitch_cmd = 0.0
    a_pitch_ach = 0.0
    alpha_lag   = 1.0 - np.exp(-dt / tau_ap)

    u_warm    = None
    t         = 0.0
    step      = 0
    mpc_interval = mpc_dt
    time_since_guidance = mpc_interval

    prev_range = np.inf
    miss_distance = np.inf

    while t < t_max:
        R_vec = r_T - r_M
        R     = float(np.linalg.norm(R_vec))

        if R < miss_distance:
            miss_distance = R

        # 종료 조건
        if R < 5.0:
            break
        if R > prev_range * 1.05 and t > 2.0:
            break
        if r_M[2] > 0:  # 지면 충돌 (NED: z>0 이면 지면 아래)
            break

        prev_range = R

        # LOS 상태
        x_los = ned_to_los_state(r_M, v_M, r_T, v_T, a_pitch_ach, 0.0)

        # 유도 명령 갱신
        time_since_guidance += dt
        if time_since_guidance >= mpc_interval:
            time_since_guidance = 0.0

            Vc = x_los[1]
            if guidance == 'mpc':
                if Vc > 1.0:
                    t_go = R / Vc
                    N_eff = min(mpc_N, max(2, int(t_go / mpc_dt)))
                else:
                    N_eff = mpc_N

                if N_eff >= 2:
                    a_p, _, u_warm_new = mpc_step(
                        x_los, N=N_eff, dt_mpc=mpc_dt, tau_ap=tau_ap,
                        a_max=a_max,
                        w_terminal=mpc_w_terminal, w_effort=mpc_w_effort,
                        w_rate=mpc_w_rate, u_warm=u_warm
                    )
                    a_pitch_cmd = float(np.clip(a_p, -a_max, a_max))
                    if u_warm_new.shape[0] > 1:
                        u_warm = np.vstack([u_warm_new[1:], u_warm_new[-1:]])
                    else:
                        u_warm = None
                else:
                    # PN fallback
                    lam_dot_el = x_los[3]
                    a_pitch_cmd = float(np.clip(pn_N * Vc * lam_dot_el, -a_max, a_max))

            else:  # APN
                lam_dot_el = x_los[3]
                a_pitch_cmd = float(np.clip(pn_N * max(Vc, 0.0) * lam_dot_el, -a_max, a_max))

        # 달성 가속도 (1차 지연)
        a_pitch_ach += alpha_lag * (a_pitch_cmd - a_pitch_ach)

        # 미사일 RK4 적분 (포인트 매스, 2D)
        def deriv_M(r, v):
            ax = 0.0  # drag 무시 (단순화)
            az = -a_pitch_ach  # NED: Down 방향 가속 (음수 = 상승)
            return v, np.array([ax, 0.0, az])

        def deriv_T(r, v):
            return v, np.zeros(3)

        def rk4_3d(r, v, deriv_fn):
            v1, a1 = deriv_fn(r, v)
            v2, a2 = deriv_fn(r + 0.5*dt*v1, v + 0.5*dt*a1)
            v3, a3 = deriv_fn(r + 0.5*dt*v2, v + 0.5*dt*a2)
            v4, a4 = deriv_fn(r +     dt*v3, v +     dt*a3)
            r_new  = r + (dt/6)*(v1 + 2*v2 + 2*v3 + v4)
            v_new  = v + (dt/6)*(a1 + 2*a2 + 2*a3 + a4)
            return r_new, v_new

        r_M, v_M = rk4_3d(r_M, v_M, deriv_M)
        r_T, v_T = rk4_3d(r_T, v_T, deriv_T)

        # 기록 (매 5스텝)
        if step % 5 == 0:
            t_hist.append(t)
            rM_hist.append(r_M.copy())
            rT_hist.append(r_T.copy())
            acmd_hist.append(a_pitch_cmd)
            range_hist.append(R)
            lam_dot_hist.append(x_los[3])

        t    += dt
        step += 1

    return {
        't'           : np.array(t_hist),
        'r_M'         : np.array(rM_hist),
        'r_T'         : np.array(rT_hist),
        'a_cmd'       : np.array(acmd_hist),
        'range'       : np.array(range_hist),
        'lam_dot'     : np.array(lam_dot_hist),
        'miss_distance': miss_distance,
        'tof'         : t,
    }


# ─────────────────────────────────────────────────────────────────
# 시뮬레이션 실행 (MPC)
# ─────────────────────────────────────────────────────────────────
print("MPC 교전 시뮬레이션 실행 중... (약 10-30초 소요)")

res_mpc = run_2d_engagement_mpc(
    guidance='mpc',
    mpc_N=10, mpc_dt=0.1,
    mpc_w_terminal=100.0, mpc_w_effort=0.0, mpc_w_rate=0.005,
    a_max=392.4, tau_ap=0.05
)

print(f"MPC 결과:")
print(f"  비행 시간: {res_mpc['tof']:.2f}초")
print(f"  Miss distance: {res_mpc['miss_distance']:.2f}m")

# 궤적, 가속도, LOS rate 플롯
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
if len(res_mpc['r_M']) > 0:
    ax.plot(res_mpc['r_M'][:, 0]/1000, -res_mpc['r_M'][:, 2]/1000, 'b-', lw=2, label='유도탄 (MPC)')
    ax.plot(res_mpc['r_T'][:, 0]/1000, -res_mpc['r_T'][:, 2]/1000, 'r--', lw=2, label='표적')
    ax.plot(res_mpc['r_M'][0, 0]/1000, -res_mpc['r_M'][0, 2]/1000, 'bs', ms=8)
    ax.plot(res_mpc['r_T'][0, 0]/1000, -res_mpc['r_T'][0, 2]/1000, 'rs', ms=8)
ax.set_xlabel('North (km)')
ax.set_ylabel('고도 (km)')
ax.set_title('비행 궤적 (MPC)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

ax = axes[1]
if len(res_mpc['t']) > 0:
    ax.plot(res_mpc['t'], res_mpc['a_cmd']/9.81, 'b-', lw=1.5)
    ax.axhline(y=40, color='r', ls='--', alpha=0.7, label='+40g 한계')
    ax.axhline(y=-40, color='r', ls='--', alpha=0.7, label='-40g 한계')
ax.set_xlabel('시간 (s)')
ax.set_ylabel('가속도 명령 (g)')
ax.set_title('가속도 명령 (MPC)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
if len(res_mpc['t']) > 0:
    ax.plot(res_mpc['t'], np.degrees(res_mpc['lam_dot']), 'b-', lw=1.5)
ax.set_xlabel('시간 (s)')
ax.set_ylabel('LOS 각속도 (deg/s)')
ax.set_title('LOS Rate (MPC)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. MPC vs PN 비교

동일한 교전 시나리오에서 Augmented PN(APN)과 MPC를 비교한다.

**비교 포인트**:
- 가속도 사용 효율
- LOS rate 수렴 특성
- Miss distance

> **가설**: MPC는 미래를 예측하므로 가속도를 더 '계획적으로' 사용할 것이다.  
> PN은 현재 LOS rate에 반응적으로 명령을 내린다.

In [ ]:
"""Cell 4: MPC vs APN 비교"""

print("APN 교전 시뮬레이션 실행 중...")

res_apn = run_2d_engagement_mpc(
    guidance='apn',
    pn_N=4.0,
    a_max=392.4, tau_ap=0.05
)

print(f"APN 결과:")
print(f"  비행 시간: {res_apn['tof']:.2f}초")
print(f"  Miss distance: {res_apn['miss_distance']:.2f}m")
print()
print(f"비교 요약:")
print(f"  APN miss: {res_apn['miss_distance']:.2f}m  /  MPC miss: {res_mpc['miss_distance']:.2f}m")

# ─────────────────────────────────────────────────────────────────
# 나란히 비교 플롯
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = {'mpc': 'steelblue', 'apn': 'darkorange'}

# 궤적 비교
ax = axes[0]
for label, res, c in [('MPC', res_mpc, colors['mpc']), ('APN (N=4)', res_apn, colors['apn'])]:
    if len(res['r_M']) > 0:
        ax.plot(res['r_M'][:, 0]/1000, -res['r_M'][:, 2]/1000,
                '-', color=c, lw=2, label=f'유도탄 {label}')
        ax.plot(res['r_M'][0, 0]/1000, -res['r_M'][0, 2]/1000, 's', color=c, ms=8)
if len(res_mpc['r_T']) > 0:
    ax.plot(res_mpc['r_T'][:, 0]/1000, -res_mpc['r_T'][:, 2]/1000,
            'r--', lw=2, label='표적')
    ax.plot(res_mpc['r_T'][0, 0]/1000, -res_mpc['r_T'][0, 2]/1000, 'rs', ms=8)
ax.set_xlabel('North (km)')
ax.set_ylabel('고도 (km)')
ax.set_title('비행 궤적 비교')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 가속도 크기 비교
ax = axes[1]
for label, res, c in [('MPC', res_mpc, colors['mpc']), ('APN (N=4)', res_apn, colors['apn'])]:
    if len(res['t']) > 0:
        ax.plot(res['t'], np.abs(res['a_cmd'])/9.81,
                '-', color=c, lw=1.5, label=label)
ax.axhline(y=40, color='r', ls='--', alpha=0.5, label='40g 한계')
ax.set_xlabel('시간 (s)')
ax.set_ylabel('|가속도 명령| (g)')
ax.set_title('가속도 크기 비교')
ax.legend()
ax.grid(True, alpha=0.3)

# LOS rate 비교
ax = axes[2]
for label, res, c in [('MPC', res_mpc, colors['mpc']), ('APN (N=4)', res_apn, colors['apn'])]:
    if len(res['t']) > 0:
        ax.plot(res['t'], np.degrees(np.abs(res['lam_dot'])),
                '-', color=c, lw=1.5, label=label)
ax.set_xlabel('시간 (s)')
ax.set_ylabel('|LOS rate| (deg/s)')
ax.set_title('LOS Rate 수렴 비교')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('MPC vs APN 비교 (a_max=40g, 비기동 표적)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. 제약조건 처리의 장점

### 5.1 가속도 제한이 타이트한 경우

가속도 한계를 40g → 20g로 줄였을 때 두 유도법칙의 차이가 뚜렷해진다.

**PN의 문제**: 제약조건을 인식하지 못하고 포화(saturation)된 명령을 그대로 출력.  
터미널 페이즈에서 큰 LOS rate를 따라잡으려다 가속도가 포화되면 miss가 커진다.

**MPC의 장점**: 가속도 한계 $|u| \leq 20g$를 최적화 문제에 명시적으로 포함.  
미리 제약조건을 고려해 예측 구간 동안 '계획적으로' 가속도를 배분.

> "가속도 제한이 있는 경우 MPC의 진가가 발휘된다."  
> 이 시뮬레이션을 보고 MPC의 가치를 실감했다.

In [ ]:
"""Cell 5: 제약 조건 타이트 시나리오 (20g 한계)"""

A_MAX_TIGHT = 20.0 * 9.81  # 20g (타이트한 제한)

print(f"가속도 한계 {A_MAX_TIGHT/9.81:.0f}g 시나리오 실행 중...")
print("APN (20g limit) 실행...")

res_apn_tight = run_2d_engagement_mpc(
    guidance='apn',
    pn_N=4.0,
    a_max=A_MAX_TIGHT, tau_ap=0.05
)

print("MPC (20g limit) 실행...")

res_mpc_tight = run_2d_engagement_mpc(
    guidance='mpc',
    mpc_N=10, mpc_dt=0.1,
    mpc_w_terminal=100.0, mpc_w_effort=0.0, mpc_w_rate=0.005,
    a_max=A_MAX_TIGHT, tau_ap=0.05
)

print()
print("=" * 50)
print(f"  {'':20s}  {'40g 한계':>12s}  {'20g 한계':>12s}")
print("-" * 50)
print(f"  {'APN miss (m)':20s}  {res_apn['miss_distance']:12.2f}  {res_apn_tight['miss_distance']:12.2f}")
print(f"  {'MPC miss (m)':20s}  {res_mpc['miss_distance']:12.2f}  {res_mpc_tight['miss_distance']:12.2f}")
print("=" * 50)

# ─────────────────────────────────────────────────────────────────
# 비교 플롯
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenarios = [
    ('APN, 40g', res_apn,       'darkorange', '-'),
    ('APN, 20g', res_apn_tight, 'darkorange', '--'),
    ('MPC, 40g', res_mpc,       'steelblue',  '-'),
    ('MPC, 20g', res_mpc_tight, 'steelblue',  '--'),
]

# 궤적
ax = axes[0, 0]
for lbl, res, c, ls in scenarios:
    if len(res['r_M']) > 0:
        ax.plot(res['r_M'][:, 0]/1000, -res['r_M'][:, 2]/1000,
                ls=ls, color=c, lw=2, label=lbl)
if len(res_mpc['r_T']) > 0:
    ax.plot(res_mpc['r_T'][:, 0]/1000, -res_mpc['r_T'][:, 2]/1000,
            'r--', lw=1.5, alpha=0.7, label='표적')
ax.set_xlabel('North (km)')
ax.set_ylabel('고도 (km)')
ax.set_title('비행 궤적 (4가지 시나리오)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 가속도 (40g 기준선)
ax = axes[0, 1]
for lbl, res, c, ls in scenarios:
    if len(res['t']) > 0:
        ax.plot(res['t'], np.abs(res['a_cmd'])/9.81,
                ls=ls, color=c, lw=1.5, label=lbl)
ax.axhline(y=40, color='gray', ls=':', alpha=0.6, label='40g 한계')
ax.axhline(y=20, color='red',  ls=':', alpha=0.6, label='20g 한계')
ax.set_xlabel('시간 (s)')
ax.set_ylabel('|가속도 명령| (g)')
ax.set_title('가속도 명령 비교')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# LOS rate
ax = axes[1, 0]
for lbl, res, c, ls in scenarios:
    if len(res['t']) > 0:
        ax.plot(res['t'], np.degrees(np.abs(res['lam_dot'])),
                ls=ls, color=c, lw=1.5, label=lbl)
ax.set_xlabel('시간 (s)')
ax.set_ylabel('|LOS rate| (deg/s)')
ax.set_title('LOS Rate 수렴 비교')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Miss distance 막대 차트
ax = axes[1, 1]
bar_labels = ['APN\n40g', 'APN\n20g', 'MPC\n40g', 'MPC\n20g']
bar_vals   = [
    res_apn['miss_distance'],
    res_apn_tight['miss_distance'],
    res_mpc['miss_distance'],
    res_mpc_tight['miss_distance'],
]
bar_colors = ['darkorange', 'darkorange', 'steelblue', 'steelblue']
bar_alpha   = [1.0, 0.5, 1.0, 0.5]
bars = ax.bar(bar_labels, bar_vals, color=bar_colors,
               alpha=0.8, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, bar_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}m', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Miss Distance (m)')
ax.set_title('Miss Distance 비교')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(bar_vals)*1.3 + 5)

plt.suptitle('제약조건 처리 비교: APN vs MPC (40g vs 20g)', fontsize=13)
plt.tight_layout()
plt.show()

print()
print("관찰: 가속도 제한이 타이트할수록 MPC와 APN의 차이가 더 뚜렷해진다.")
print("MPC는 제약조건 내에서 최적으로 가속도를 배분하기 때문이다.")

---
## 6. 한계와 향후 과제

### 6.1 계산 시간 문제

위 scipy 기반 구현의 가장 큰 문제는 **계산 시간**이다.

| 구현 | 스텝당 계산 시간 | 비고 |
|------|-----------------|------|
| scipy L-BFGS-B | ~수십~수백ms | 실시간 불가 |
| CasADi + IPOPT | ~1~5ms | 실시간 가능성 있음 |
| 임베디드 QP solver (e.g., qpOASES) | ~0.1~1ms | 임베디드 실시간 |

유도탄 탑재 컴퓨터에서 $dt = 10\,\text{ms}$ 내에 풀어야 하므로, 계산 시간은 결정적인 제약이다.

### 6.2 모델 불확실성

현재 MPC는 완벽한 모델을 가정한다:
- 표적 가속도 $a_T$를 정확히 안다
- 자동조종장치 동특성 $\tau_{ap}$를 정확히 안다
- 공기역학 불확실성 없음

실제로는 모델 오차와 외란이 존재하므로 **Robust MPC** 또는 **Tube MPC**가 필요하다.

### 6.3 향후 연구 방향

1. **CasADi + IPOPT**: 자동 미분으로 gradient 계산 → 실시간 가능성 검토
   ```python
   # CasADi 기반 구현 (별도 파일: mpc_guidance.py)
   import casadi as ca
   solver = ca.nlpsol('mpc', 'ipopt', nlp, opts)
   ```

2. **Convex Relaxation**: 비선형 문제를 볼록 문제로 근사 (Boyd & Vandenberghe 참고)  
   ZEM 기반 선형화 → QP 문제로 변환 → 빠른 해법

3. **임베디드 QP 솔버**: qpOASES, OSQP 등 임베디드 환경용 솔버

4. **GP 보정**: 모델 불확실성을 Gaussian Process로 학습하고 MPC에 피드백  
   (이 부분은 별도 노트북에서 연구)

> "CasADi + IPOPT를 사용하면 실시간 가능성 있음 — 추가 연구 필요"  
> 실제로 프로젝트에서 CasADi 기반 구현을 시도했고, 약 2~5ms 수준의 계산 시간을 달성했다.  
> 하지만 warm start 없이는 수렴이 불안정한 경우가 있었다.

---
## 7. 정리

### 7.1 요약

이 노트북에서 탐구한 내용:

| 항목 | 내용 |
|------|------|
| LOS 동역학 | Zarchan Ch.8 횡방향 방정식, 부호 규약 확인 |
| MPC 정식화 | ZEM 기반 비용 함수, shooting method |
| scipy 구현 | L-BFGS-B, bound constraints |
| 가중치 튜닝 | $w_{effort}=0$, $w_{terminal}=100$ 최적 |
| 비교 실험 | MPC vs APN, 40g vs 20g 한계 |

### 7.2 배운 점

**비례항법이 여전히 실전 표준인 이유를 체감했다.**

MPC는 이론적으로 우월하지만:
- 계산 부담이 크다 (임베디드 환경에서 문제)
- 모델 의존성이 높다 (표적 기동 추정, 동특성 정확도)
- 구현 복잡도가 높다 (NLP 솔버, warm start 관리)
- 튜닝 파라미터가 많다 (N, dt_mpc, 가중치)

반면 PN은:
- 연산량이 거의 없다 (곱셈 몇 번)
- 강인하다 (모델 불확실성에 덜 민감)
- 검증된 실적이 있다 (수십 년 비행 시험)

### 7.3 향후

> "향후 convex optimization 기반 접근도 연구해보고 싶다."

Boyd의 *Convex Optimization*에서 다루는 볼록 완화 기법을 적용하면,  
비선형 MPC 문제를 QP(이차 계획)로 변환해 계산 시간을 크게 줄일 수 있다.  
또한 LASSO/Ridge 형태의 정규화로 robust성을 높이는 방향도 흥미롭다.

---

*개인 연구 노트 — 석사 과정 중 MPC 유도 탐구*  
*Reference: Zarchan (2019) Ch.8 / Boyd & Vandenberghe, Convex Optimization (2004)*